In [40]:
import pickle
import itertools
from dataclasses import dataclass, is_dataclass, asdict
from typing import List, Dict

import pandas
import numpy as np
from torch.utils.data import DataLoader

from disk_analyzer.models import DiskDataset
from disk_analyzer.stages.model_scoring import ModelScorer

In [41]:
EXP_NUM = 111
DATA_FOLDER = "Data/Preprocessed_long"
BASE_DIR = f"Artifacts/Exp_{EXP_NUM}"
TIMES = np.arange(0, 2095)
TRAIN_TIMES = TIMES[::20]
TRAIN_BATCHSIZE = 1000
TRAIN_SAMPLES = 20

@dataclass
class DataConfig:
    data_folder: str
    train_batchsize: int
    score_batchsize: int
    times: np.ndarray
    train_times: np.ndarray
    to_cens_shift: List[int]
    to_term_shift: List[int]
    cens_prob: float

@dataclass
class ExperimentConfig:
    metrics: List[str]
    schema: Dict[str, list]
    res_filename: str
    models_folder: str
    log_dir: str

data_cfg = DataConfig(
    data_folder=DATA_FOLDER,
    train_batchsize=TRAIN_BATCHSIZE,
    score_batchsize=512,
    times=TRAIN_TIMES,
    train_times=TRAIN_TIMES,
    to_cens_shift=[],
    to_term_shift=[],
    cens_prob=-1,
)

exp_cfg = ExperimentConfig(
    metrics=['ci', 'ibs'],
    schema=None,
    res_filename=None,
    models_folder=None,
    log_dir=None,
)

In [42]:
with open('Artifacts/Exp_111/models/0_SP.pkl', 'rb') as f:
    model = pickle.load(f)

In [43]:
def prepare_dataloader(
    train_samples: int,
    data_cfg,
    data_type: str,
    dataset_type: str,
    test_samples,
    data_ext: str = "csv"  # "csv "or "parquet"
):
    if data_type == "train":
        files = [f"{data_cfg.data_folder}/{train_samples}_train_preprocessed.{data_ext}"]
        batch_size = data_cfg.train_batchsize
    else:
        if test_samples is None:
            raise ValueError("test_samples must be provided for test loader")
        files = [f"{data_cfg.data_folder}/{train_samples}_{test_samples}_test_preprocessed.{data_ext}"]
        batch_size = data_cfg.score_batchsize

    ds = DiskDataset(
        dataset_type,
        files,
        to_cens_time_list=data_cfg.to_cens_shift,
        to_term_time_list=data_cfg.to_term_shift,
        cens_prob=data_cfg.cens_prob,
    )
    return DataLoader(ds, batch_size=batch_size)
dl_train = prepare_dataloader(TRAIN_SAMPLES, data_cfg, "train", "train", test_samples = None, data_ext='parquet')
limited_dataloader = itertools.islice(dl_train, 10)

In [44]:
scorer = ModelScorer()
preds, gt = model.predict(limited_dataloader, data_cfg.times)
train_metrics = scorer.get_metrics(
model, preds, gt, data_cfg.times,
metrics=exp_cfg.metrics,
df_train=None,
)

10it [00:00, 10.52it/s]


In [45]:
e_times = model.get_expected_time_by_predictions(preds, data_cfg.times)

In [46]:
np.mean(e_times)

514.6018032446752